# Functions

In [1]:
import json
import os
import random
import numpy as np
import torch as tr
from datetime import datetime
import pandas as pd
import shutil
import pickle

from torch.utils.data import DataLoader
from sincfold.dataset import SeqDataset, pad_batch
from sincfold.model import sincfold
from sincfold.embeddings import NT_DICT
from sincfold.utils import write_ct, validate_file, ct2dot
from sincfold.parser import parser
from sincfold.utils import dot2png, ct2svg

from sincfold.ablation.ablation_1ResNet2d import sincfold_1ResNet2d
from sincfold.ablation.ablation_no_ResNet1d_FF import sincfold_no_ResNet1d_FF
from sincfold.ablation.ablation_no_ResNet2d import sincfold_no_ResNet2d
from sincfold.ablation.ablation_C1D_C2D import sincfold_C1D_C2D
from sincfold.ablation.ablation_no_ResNet1d import sincfold_no_ResNet1d

In [2]:
def test(net, test_file, model_weights=None, output_file=None, config={}, nworkers=2, verbose=True):
    test_file = test_file
    test_file = validate_file(test_file)
    if verbose not in config:
        config["verbose"] = verbose

    test_loader = DataLoader(
        SeqDataset(test_file, **config),
        batch_size=config["batch_size"] if "batch_size" in config else 4,
        shuffle=False,
        num_workers=nworkers,
        collate_fn=pad_batch,
    )

    if model_weights is not None:
        net = net(weights=model_weights, **config)
    else:
        net = net(pretrained=True, **config)
    
    if verbose:
        print(f"Start test of {test_file}")        
    test_metrics = net.test(test_loader)
    summary = ",".join([k for k in sorted(test_metrics.keys())]) + "\n" + ",".join([f"{test_metrics[k]:.3f}" for k in sorted(test_metrics.keys())])+ "\n" 
    if output_file is not None:
        with open(output_file, "w") as f:
            f.write(summary)
    if verbose:
        print(summary)

In [3]:
!ls homology/data/

ls: cannot access 'homology/data/': No such file or directory


## Params and paths

In [4]:
WEIGHT_PATH = 'crossFamily/weights/'
crossFamily = os.listdir(WEIGHT_PATH)
print(crossFamily)
ablation = os.listdir(os.path.join(WEIGHT_PATH,crossFamily[0]))
print(ablation)

DATA_PATH = 'crossFamily/data/'

test_files = [f for f in os.listdir(DATA_PATH) if f.startswith('test')]
print(test_files)

['tRNA', 'RNaseP', 'telomerase']
['C1D_C2D', 'no_ResNet_2d']
['test_tRNA.csv', 'test_RNaseP.csv', 'test_telomerase.csv']


In [5]:
print(crossFamily[0], ablation[1])

tRNA no_ResNet_2d


In [6]:
output = os.path.join(os.path.join(WEIGHT_PATH,crossFamily[0]), ablation[1]) + f'/test_{crossFamily[0]}.csv'
output

'crossFamily/weights/tRNA/no_ResNet_2d/test_tRNA.csv'

# Ablations

## No ResNet 2d

In [7]:
test_file =test_files[0] 
test_path = os.path.join(DATA_PATH, test_file)
model_weights = os.path.join(os.path.join(WEIGHT_PATH,crossFamily[0]), ablation[1]) + '/weights.pmt'
output = os.path.join(os.path.join(WEIGHT_PATH,crossFamily[0]), ablation[1]) + f'/test_{crossFamily[0]}.csv'

print(f"test_file: {test_file}")
print(f"test_path: {test_path}")
print(f"model_weights: {model_weights}")
print(f"output: {output}")

test_file: test_tRNA.csv
test_path: crossFamily/data/test_tRNA.csv
model_weights: crossFamily/weights/tRNA/no_ResNet_2d/weights.pmt
output: crossFamily/weights/tRNA/no_ResNet_2d/test_tRNA.csv


In [8]:
# Ejecutar el test para cada archivo de prueba
test(
    net=sincfold_no_ResNet2d, 
    test_file=test_path,     
    model_weights= model_weights,  
    output_file= output,  
    nworkers=2, 
    verbose=True
)

Load weights from crossFamily/weights/tRNA/no_ResNet_2d/weights.pmt
Start test of crossFamily/data/test_tRNA.csv


100%|██████████| 140/140 [00:28<00:00,  4.89it/s]

f1,f1_post,loss
0.546,0.559,0.096



In [9]:
test_file =test_files[1] 
test_path = os.path.join(DATA_PATH, test_file)
model_weights = os.path.join(os.path.join(WEIGHT_PATH,crossFamily[1]), ablation[1]) + '/weights.pmt'
output = os.path.join(os.path.join(WEIGHT_PATH,crossFamily[1]), ablation[1]) + f'/test_{crossFamily[1]}.csv'

print(f"test_file: {test_file}")
print(f"test_path: {test_path}")
print(f"model_weights: {model_weights}")
print(f"output: {output}")

test_file: test_RNaseP.csv
test_path: crossFamily/data/test_RNaseP.csv
model_weights: crossFamily/weights/RNaseP/no_ResNet_2d/weights.pmt
output: crossFamily/weights/RNaseP/no_ResNet_2d/test_RNaseP.csv


In [10]:
# Ejecutar el test para cada archivo de prueba
test(
    net=sincfold_no_ResNet2d,  
    test_file=test_path,     
    model_weights= model_weights,  
    output_file= output,  
    nworkers=2,  
    verbose=True
)

Load weights from crossFamily/weights/RNaseP/no_ResNet_2d/weights.pmt
Start test of crossFamily/data/test_RNaseP.csv


100%|██████████| 114/114 [04:39<00:00,  2.45s/it]

f1,f1_post,loss
0.238,0.271,0.040



In [11]:
test_file =test_files[2] 
test_path = os.path.join(DATA_PATH, test_file)
model_weights = os.path.join(os.path.join(WEIGHT_PATH,crossFamily[2]), ablation[1]) + '/weights.pmt'
output = os.path.join(os.path.join(WEIGHT_PATH,crossFamily[2]), ablation[1]) + f'/test_{crossFamily[2]}.csv'

print(f"test_file: {test_file}")
print(f"test_path: {test_path}")
print(f"model_weights: {model_weights}")
print(f"output: {output}")

test_file: test_telomerase.csv
test_path: crossFamily/data/test_telomerase.csv
model_weights: crossFamily/weights/telomerase/no_ResNet_2d/weights.pmt
output: crossFamily/weights/telomerase/no_ResNet_2d/test_telomerase.csv


In [12]:
# Ejecutar el test para cada archivo de prueba
test(
    net=sincfold_no_ResNet2d,  
    test_file=test_path,     
    model_weights= model_weights,  
    output_file= output,  
    nworkers=2,  
    verbose=True
)

Load weights from crossFamily/weights/telomerase/no_ResNet_2d/weights.pmt
Start test of crossFamily/data/test_telomerase.csv


  0%|          | 0/9 [00:00<?, ?it/s]

100%|██████████| 9/9 [00:35<00:00,  3.90s/it]

f1,f1_post,loss
0.130,0.158,0.034



## C1D_C2D

In [13]:
for i in range(3):
    test_file =test_files[i]
    test_path = os.path.join(DATA_PATH, test_file)
    model_weights = os.path.join(os.path.join(WEIGHT_PATH,crossFamily[i]), ablation[0]) + '/weights.pmt'
    output = os.path.join(os.path.join(WEIGHT_PATH,crossFamily[i]), ablation[0]) + f'/test_{crossFamily[i]}.csv'

    print(f"test_file: {test_file}")
    print(f"test_path: {test_path}")
    print(f"model_weights: {model_weights}")
    print(f"output: {output}")
    # Ejecutar el test para cada archivo de prueba
    test(
        net=sincfold_C1D_C2D,  
        test_file=test_path,     
        model_weights= model_weights,  
        output_file= output,  
        nworkers=2,  
        verbose=True
    )

test_file: test_tRNA.csv
test_path: crossFamily/data/test_tRNA.csv
model_weights: crossFamily/weights/tRNA/C1D_C2D/weights.pmt
output: crossFamily/weights/tRNA/C1D_C2D/test_tRNA.csv
Load weights from crossFamily/weights/tRNA/C1D_C2D/weights.pmt
Start test of crossFamily/data/test_tRNA.csv


  0%|          | 0/140 [00:00<?, ?it/s]

100%|██████████| 140/140 [00:15<00:00,  8.86it/s]


f1,f1_post,loss
0.500,0.508,0.095

test_file: test_RNaseP.csv
test_path: crossFamily/data/test_RNaseP.csv
model_weights: crossFamily/weights/RNaseP/C1D_C2D/weights.pmt
output: crossFamily/weights/RNaseP/C1D_C2D/test_RNaseP.csv
Load weights from crossFamily/weights/RNaseP/C1D_C2D/weights.pmt
Start test of crossFamily/data/test_RNaseP.csv


100%|██████████| 114/114 [04:21<00:00,  2.30s/it]


f1,f1_post,loss
0.269,0.287,0.037

test_file: test_telomerase.csv
test_path: crossFamily/data/test_telomerase.csv
model_weights: crossFamily/weights/telomerase/C1D_C2D/weights.pmt
output: crossFamily/weights/telomerase/C1D_C2D/test_telomerase.csv
Load weights from crossFamily/weights/telomerase/C1D_C2D/weights.pmt
Start test of crossFamily/data/test_telomerase.csv


100%|██████████| 9/9 [00:34<00:00,  3.83s/it]

f1,f1_post,loss
0.152,0.184,0.030



# SINCFOLD Test

In [14]:
test_files

['test_tRNA.csv', 'test_RNaseP.csv', 'test_telomerase.csv']

In [ ]:
for i in range(3):
    test_file =test_files[i]
    test_path = os.path.join(DATA_PATH, test_file)
    model_weights = '/weights.pmt'
    output = os.path.join(os.path.join(WEIGHT_PATH,crossFamily[i]), ablation[0]) + f'/test_{crossFamily[i]}.csv'

    print(f"test_file: {test_file}")
    print(f"test_path: {test_path}")
    print(f"model_weights: {model_weights}")
    print(f"output: {output}")
    # Ejecutar el test para cada archivo de prueba
    test(
        net=sincfold_C1D_C2D,  
        test_file=test_path,     
        model_weights= model_weights,  
        output_file= output,  
        nworkers=2,  
        verbose=True
    )